In [20]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv

# 1. Load the .env from the parent directory
# find_dotenv() searches for the file; you can also use: load_dotenv('../.env')
load_dotenv(find_dotenv())

# 2. Fetch variables from the environment
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# 3. Create the Connection String
# Using an f-string makes this much cleaner
db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

# 2. Read the local CSV
# We only load the columns we need to save memory
file_path = '../data/equipment.csv'
cols_to_import = ['ip_address', 'category', 'type', 'plaza_id', 'lane_id']
df = pd.read_csv(file_path, usecols=cols_to_import)



In [23]:
# check connection to the database
try:
    # This will return a tiny dataframe with a single 1
    test_df = pd.read_sql("SELECT 1 as connection_test", engine)
    print("Connection Successful!")
    print(test_df)
except Exception as e:
    print(f"Connection Failed: {e}")

Connection Successful!
   connection_test
0                1


In [21]:
import os
print(f"File exists: {os.path.exists(file_path)}")
print(f"File size: {os.path.getsize(file_path)} bytes")

File exists: True
File size: 249433 bytes


In [22]:
# Check the shape (Rows, Columns)
print(f"Rows found: {df.shape[0]}")
print(f"Columns found: {df.shape[1]}")

# Show the first few rows of the RAW data
df.head()

Rows found: 1228
Columns found: 5


,ip_address,category,type,plaza_id,lane_id
0,10.0.79.153,side-firing,camera,466,M03
1,10.0.79.156,side-firing,camera,466,M06
2,10.0.79.151,side-firing,camera,466,M01
3,10.0.84.151,side-firing,camera,472,M01
4,10.0.83.153,side-firing,camera,471,M03


In [ ]:
# filtered data
filtered_df = df[df['type'] == 'camera'][['ip_address', 'category', 'type', 'plaza_id', 'lane_id']]

# Shows column names, non-null counts, and data types (int64, float64, object, etc.)
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 1050 entries, 0 to 1225
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ip_address  1050 non-null   str  
 1   category    1050 non-null   str  
 2   type        1050 non-null   str  
 3   plaza_id    1050 non-null   str  
 4   lane_id     1050 non-null   str  
dtypes: str(5)
memory usage: 49.2 KB


In [18]:
# 4. Insert into PostgreSQL
# 'if_exists' can be 'append', 'replace', or 'fail'
table_name = 'tmp_cam_equipment'
filtered_df.to_sql(table_name, engine, if_exists='append', index=False)

print(f"Successfully inserted {len(filtered_df)} rows into {table_name}.")

Successfully inserted 1050 rows into tmp_cam_equipment.
